# bcdl-ids on Colab — train and look inside

This notebook **does not change the bcdl-ids repository**. It clones it read-only and *imports* its code
(`BCSimpleCNN`, `BCDLDataset`, `train_one_epoch`, `evaluate`, `trace`, ...), then adds pictures on top.

Before running: **Runtime → Change runtime type → T4 GPU** (CPU also works, just slower).

| Cell | What it does |
|---|---|
| 1. Setup | mounts Drive, clones/pulls the repo, copies `bcdl-faces` to fast local disk |
| 2. Imports | imports the repo modules, prints device and the two CSV files |
| 3. Helpers | small plotting functions used by every later cell |
| 4. Dataset | `samples`, train/val split, a grid of the raw faces |
| 5. Transforms | one image through the transform, step by step; 6 random augmentations |
| 6. One batch | the `[B, 3, 224, 224]` tensor the network actually receives |
| 7. Model | layers and parameter counts |
| 8. Untrained trace | layer-by-layer table, feature maps, conv1 filters with random weights |
| 9. One training step | logits → softmax → loss → gradients, every value printed |
| 10. Training | 30 epochs, history recorded, best checkpoint saved to Drive |
| 11. Curves | loss / accuracy / learning rate per epoch |
| 12. Trained trace | same image, trained weights: maps, filters, vectors, before/after statistics |
| 13. Validation | every validation face with prediction, confusion matrix |
| 14. Explorer | sliders: pick any image, any layer, see its feature maps |

## 1. Setup

* The repo is cloned to `/content/bcdl-ids` (re-running the cell does `git pull` instead).
* The dataset is **copied** from Drive to `/content/bcdl-faces`. Reading many small files through the Drive
  mount is slow; the local copy makes every epoch read from the VM's own disk.
* `BCDL_DATA_DIR` is the environment variable `bcdl_defs.py` already understands — it must be set
  **before** `bcdl_defs` is imported.
* `RUN_DIR` is on Drive, so the checkpoint and figures survive when the Colab VM is recycled.

In [ ]:
# --- settings: change these if your folders differ ---------------------------
REPO_URL   = "https://github.com/bchavdarov/bcdl-ids.git"
REPO_DIR   = "/content/bcdl-ids"                                    # read-only copy of the repo
DRIVE_DATA = "/content/drive/MyDrive/NVIDIA/jetson-cv/bcdl-faces"   # dataset on Drive
LOCAL_DATA = "/content/bcdl-faces"                                  # fast local copy
RUN_DIR    = "/content/drive/MyDrive/NVIDIA/jetson-cv/bcdl-ids-colab"  # checkpoint + figures

import os
import shutil
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
print("repo:", subprocess.check_output(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], text=True).strip())

shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
print("dataset files:", len(os.listdir(LOCAL_DATA)), "->", LOCAL_DATA)

os.environ["BCDL_DATA_DIR"] = LOCAL_DATA       # read by bcdl_defs at import time
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 2. Imports from the repo

Everything below comes from bcdl-ids unchanged. `bcdl_trace.trace()` is reused for the layer table, so the
numbers here are the same ones you get on the Nano with `python3 bcdl_trace.py`.

The two CSV files are printed: `annotations.csv` must be `imagename,label` and `identities.csv` must be
`label,name`. If the Drive copy still has the old `imagename,person` format, upload the current files first.

In [ ]:
import contextlib
import io
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image

from bcdl_defs import (ANNOTATIONS_CSV, DATA_DIR, IDENTITIES_CSV, IMAGE_SIZE, MEAN, STD,
                       evaluate, get_device, load_identities, load_image)
from bcdl_data import build_dataloaders, build_transform, read_annotations, split_samples
from bcdl_model import BCSimpleCNN
from bcdl_train import train_one_epoch
from bcdl_inference import load_model, predict
from bcdl_trace import short, trace

device = get_device()
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU")
print("DATA_DIR:", DATA_DIR)
for p in (ANNOTATIONS_CSV, IDENTITIES_CSV):
    with open(str(p)) as f:
        print("\n{}:\n{}".format(p.name, "".join(f.readlines()[:4])), end="")

## 3. Helpers (plotting only)

| Function | What it does |
|---|---|
| `describe(name, v)` | one line per variable: shape, dtype, device, min / max / mean / std (or plain value) |
| `to_rgb(t)` | normalized `[3,H,W]` tensor → displayable `[H,W,3]` image (undoes `Normalize`) |
| `show_grid(images, titles)` | any list of images in a grid |
| `quiet_trace(model, x)` | `bcdl_trace.trace()` with its printed table suppressed |
| `show_map_rows(outputs)` | one row per layer, first *n* channels — shows 224 → 112 → 56 → 28 → 4 |
| `show_conv1_filters(model)` | the 32 learned 3×3 RGB filters of the first layer |
| `show_vectors(outputs)` | the 1-D tensors in the classifier (2048 → 256 → 2) as bar charts |
| `layer_stats(outputs)` | mean and std of every layer's output, for before/after comparison |

In [ ]:
MEAN_T = torch.tensor(MEAN).view(3, 1, 1)
STD_T = torch.tensor(STD).view(3, 1, 1)
names = load_identities()                    # {0: 'Al Pacino', 1: 'Robert De Niro'}


def describe(name, v):
    # Print shape/dtype/device/stats of a tensor, or type and value of anything else.
    if torch.is_tensor(v):
        f = v.detach().float()
        std = f.std().item() if f.numel() > 1 else 0.0
        print("{:<16} {:<20} {:<14} {:<7} min {:8.3f}  max {:8.3f}  mean {:8.3f}  std {:7.3f}".format(
            name, str(list(v.shape)), str(v.dtype), str(v.device),
            f.min().item(), f.max().item(), f.mean().item(), std))
    else:
        print("{:<16} {:<10} {!r}".format(name, type(v).__name__, v))


def to_rgb(t):
    # Normalized [3,H,W] tensor -> [H,W,3] numpy image in 0..1.
    return (t.detach().cpu() * STD_T + MEAN_T).clamp(0, 1).permute(1, 2, 0).numpy()


def show_grid(images, titles=None, cols=8, size=2.0, cmap=None, colors=None, suptitle=None):
    # Plot a list of HxW or HxWx3 images; returns the figure.
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for k, img in enumerate(images):
        ax = axes.flat[k]
        ax.imshow(img, cmap=cmap, interpolation="nearest")
        if titles:
            ax.set_title(titles[k], fontsize=8, color=colors[k] if colors else "black")
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()
    return fig


def quiet_trace(model, x):
    # bcdl_trace.trace() without the printed table: [(index, layer, output on CPU), ...]
    with contextlib.redirect_stdout(io.StringIO()):
        return trace(model, x, device)


def show_map_rows(outputs, n=8, kinds=(nn.ReLU, nn.MaxPool2d, nn.AdaptiveAvgPool2d), title=""):
    # One row per selected 4-D layer output, first n channels; each map has its own contrast.
    rows = [(i, layer, t) for i, layer, t in outputs if t.dim() == 4 and isinstance(layer, kinds)]
    fig, axes = plt.subplots(len(rows), n, figsize=(n * 1.4, len(rows) * 1.45), squeeze=False)
    for r, (i, layer, t) in enumerate(rows):
        for c in range(n):
            ax = axes[r, c]
            ax.set_xticks([])
            ax.set_yticks([])
            if c < t.shape[1]:
                ax.imshow(t[0, c].numpy(), cmap="viridis", interpolation="nearest")
        axes[r, 0].set_ylabel("#{} {}\n{}x{}".format(i, layer.__class__.__name__, t.shape[2], t.shape[3]),
                              fontsize=8)
    for c in range(n):
        axes[0, c].set_title("ch {}".format(c), fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_conv1_filters(model, title):
    # The first conv layer's weights [32, 3, 3, 3] as 32 tiny RGB images, on one common scale.
    w = model.features[0].weight.detach().cpu()
    w = (w - w.min()) / (w.max() - w.min())
    show_grid([f.permute(1, 2, 0).numpy() for f in w], ["f{}".format(k) for k in range(len(w))],
              cols=16, size=0.9, suptitle=title)


def show_vectors(outputs, title=""):
    # Bar chart of every 2-D output [1, N] (Flatten, Linear, ReLU, Dropout in the classifier).
    vecs = [(i, layer, t[0]) for i, layer, t in outputs if t.dim() == 2]
    fig, axes = plt.subplots(len(vecs), 1, figsize=(12, 1.6 * len(vecs)), squeeze=False)
    for ax, (i, layer, v) in zip(axes[:, 0], vecs):
        ax.bar(range(len(v)), v.numpy(), width=1.0)
        ax.set_title("#{} {}  -> {} values".format(i, short(layer), len(v)), fontsize=9, loc="left")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def layer_stats(outputs):
    # [(label, mean, std)] for every layer output.
    return [("#{} {}".format(i, layer.__class__.__name__), t.mean().item(), t.std().item())
            for i, layer, t in outputs]

## 4. Dataset as variables

`read_annotations()` turns `annotations.csv` into `samples`, a list of `(path, label)` pairs;
`split_samples()` takes 20 % of **each** person for validation with a fixed seed (same split every run).
Titles show the **original** file size — the images differ in size until the transform resizes them.

In [ ]:
samples = read_annotations()
train_samples, val_samples = split_samples(samples, val_split=0.2, seed=42)

describe("names", names)
describe("len(samples)", len(samples))
describe("samples[0]", samples[0])
describe("len(train)", len(train_samples))
describe("len(val)", len(val_samples))
for label, name in names.items():
    print("  {} {:<16} train {:3d}   val {:3d}".format(
        label, name, sum(l == label for _, l in train_samples), sum(l == label for _, l in val_samples)))

shown = samples[::max(1, len(samples) // 16)][:16]
show_grid([np.asarray(load_image(p)) for p, _ in shown],
          ["{}\n{}  {}x{}".format(os.path.basename(p), names[l], *Image.open(p).size) for p, l in shown],
          suptitle="raw faces (every {}th sample)".format(max(1, len(samples) // 16)));

## 5. One image through the transform

The validation transform (`build_transform(train=False)`) is applied **step by step** so you can see what
each step does: `Resize` (PIL image, new size) → `ToTensor` (uint8 0..255 → float 0..1, HWC → CHW)
→ `Normalize` (subtract ImageNet mean, divide by std per channel → values around −2..+2).

The training transform adds random crop / flip / colour jitter; six calls on the **same** file give six
different tensors — that is the augmentation the network sees every epoch.

In [ ]:
path, label = val_samples[0]
pil = load_image(path)
val_t = build_transform(train=False)
train_t = build_transform(train=True)

print("file:", path, "| person:", names[label])
print("{:<16} PIL {} mode {}".format("load_image", pil.size, pil.mode))
obj = pil
for step in val_t.transforms:
    obj = step(obj)
    if torch.is_tensor(obj):
        describe(step.__class__.__name__, obj)
    else:
        print("{:<16} PIL {} mode {}".format(step.__class__.__name__, obj.size, obj.mode))
x_val = obj                                   # [3, 224, 224], normalized

show_grid([np.asarray(pil), to_rgb(x_val)] + [to_rgb(train_t(pil)) for _ in range(6)],
          ["original", "val transform"] + ["train aug {}".format(k + 1) for k in range(6)],
          cols=8, size=2.2, suptitle="same face: validation vs. 6 random training versions");

## 6. One batch — what the network actually receives

`build_dataloaders()` stacks 8 augmented images into one tensor `[8, 3, 224, 224]` plus a label tensor `[8]`.
After `Normalize` the per-channel mean should be near 0 and std near 1.

In [ ]:
BATCH_SIZE = 8
train_loader, val_loader = build_dataloaders(batch_size=BATCH_SIZE, val_split=0.2, num_workers=2, seed=42)
images, labels = next(iter(train_loader))

describe("images", images)
describe("labels", labels)
print("per-channel mean (R,G,B):", [round(v, 3) for v in images.mean(dim=(0, 2, 3)).tolist()])
print("per-channel std  (R,G,B):", [round(v, 3) for v in images.std(dim=(0, 2, 3)).tolist()])
print("batches per epoch: train {} | val {}".format(len(train_loader), len(val_loader)))

show_grid([to_rgb(t) for t in images], ["{} = {}".format(l, names[l]) for l in labels.tolist()],
          suptitle="one training batch (augmented)");

## 7. The model

`BCSimpleCNN` straight from `bcdl_model.py`. The table lists every weight tensor; note that almost all
parameters sit in the first `Linear` layer (2048 → 256).

In [ ]:
CONV1_OUT = 32
SEED = 42

torch.manual_seed(SEED)
model = BCSimpleCNN(num_classes=len(names), conv1_out=CONV1_OUT).to(device)
print(model)

print("\n{:<28} {:<20} {:>10}".format("parameter", "shape", "count"))
for pname, p in model.named_parameters():
    print("{:<28} {:<20} {:>10,}".format(pname, str(list(p.shape)), p.numel()))
print("{:<49} {:>10,}".format("total", sum(p.numel() for p in model.parameters())))

## 8. Untrained network: trace one image

`trace()` from `bcdl_trace.py` runs the image layer by layer and prints shape, parameters, mean/std and
time per layer. Then the feature maps (rows = layers, columns = channels) and the 32 conv1 filters —
with **random** weights they respond to edges and colours only by chance.

In [ ]:
model.eval()
x = x_val.unsqueeze(0).to(device)               # batch of 1: [1, 3, 224, 224]
untrained_outputs = trace(model, x, device)

show_map_rows(untrained_outputs, title="UNTRAINED feature maps: " + os.path.basename(path))
show_conv1_filters(model, "conv1 filters - random initialization")

## 9. Anatomy of one training step

Everything `train_one_epoch()` does for a single batch, with each intermediate value shown:

1. forward pass → **logits** `[8, 2]` (raw scores)
2. **softmax** → probabilities per row (sum to 1)
3. **cross-entropy loss** = mean of −log(probability of the correct class); checked by hand
4. `loss.backward()` → a **gradient** for every weight; the bar chart shows each tensor's gradient size

No `optimizer.step()` here and gradients are cleared afterwards, so the weights stay unchanged
(cell 10 starts again from the same seed anyway).

In [ ]:
criterion = nn.CrossEntropyLoss()
model.train()
imgs, lbls = images.to(device), labels.to(device)

logits = model(imgs)
probs = torch.softmax(logits, dim=1)
loss = criterion(logits, lbls)
manual = -torch.log(probs[torch.arange(len(lbls)), lbls]).mean()

describe("logits", logits)
describe("probs", probs)
describe("loss", loss)
print("manual loss check: {:.6f}\n".format(manual.item()))
print("{:>3} {:>16} {:>16} {:>8}".format("#", "logits", "probabilities", "label"))
for k in range(len(lbls)):
    print("{:>3} {:>16} {:>16} {:>8}".format(k, str([round(v, 2) for v in logits[k].tolist()]),
          str([round(v, 2) for v in probs[k].tolist()]), lbls[k].item()))

model.zero_grad(set_to_none=True)
loss.backward()
grads = [(n, p.grad.norm().item()) for n, p in model.named_parameters() if p.grad is not None]
model.zero_grad(set_to_none=True)

plt.figure(figsize=(12, 3.5))
plt.bar([n for n, _ in grads], [g for _, g in grads])
plt.xticks(rotation=90, fontsize=7)
plt.ylabel("gradient norm")
plt.title("gradient size per weight tensor after one backward pass")
plt.tight_layout()
plt.show()

## 10. Training

The same loop as `bcdl_train.py` (it calls the repo's `train_one_epoch()` and `evaluate()`), with two
additions: every epoch's numbers go into `history` for plotting, and the checkpoint goes to `RUN_DIR`
on Drive. The checkpoint has **exactly the format `bcdl_train.py` writes**, so `bcdl_inference.py` and
`bcdl_trace.py` on the Nano can load it unchanged.

In [ ]:
EPOCHS, LR, WEIGHT_DECAY = 30, 1e-3, 1e-4

torch.manual_seed(SEED)                        # same start weights as the untrained model above
model = BCSimpleCNN(num_classes=len(names), conv1_out=CONV1_OUT).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

ckpt_path = os.path.join(RUN_DIR, "best_model.pt")
history = {"train_loss": [], "val_loss": [], "val_acc": [], "lr": []}
best = (-1.0, 0.0)                             # (val_acc, -val_loss): higher is better

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    history["lr"].append(optimizer.param_groups[0]["lr"])
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    mark = ""
    if (val_acc, -val_loss) > best:
        best = (val_acc, -val_loss)
        torch.save({"model_state_dict": model.state_dict(), "class_names": names,
                    "num_classes": len(names), "conv1_out": CONV1_OUT, "image_size": IMAGE_SIZE,
                    "epoch": epoch, "val_acc": val_acc}, ckpt_path)
        mark = "  <- saved"
    print("epoch {:3d}/{} | train_loss {:.4f} | val_loss {:.4f} | val_acc {:.3f} | {:.1f}s{}".format(
        epoch, EPOCHS, train_loss, val_loss, val_acc, time.time() - start, mark))

print("best val_acc {:.3f} -> {}".format(best[0], ckpt_path))

## 11. Training curves

Train loss falling while val loss rises = memorizing (overfitting). With 12 validation images each one is
≈ 8 % accuracy, so the accuracy curve moves in big steps. The right panel shows the cosine learning-rate
schedule. The figure is also saved to `RUN_DIR`.

In [ ]:
ep = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(ep, history["train_loss"], label="train")
axes[0].plot(ep, history["val_loss"], label="val")
axes[0].set_title("loss")
axes[0].legend()
axes[1].plot(ep, history["val_acc"], marker="o", markersize=3)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("validation accuracy")
axes[2].plot(ep, history["lr"])
axes[2].set_title("learning rate")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RUN_DIR, "curves.png"), dpi=120)
plt.show()

## 12. Trained network: the same image again

The best checkpoint is loaded with the repo's `load_model()` — the same call the Nano uses.
Compare with cell 8:

* **feature maps** — trained maps light up on face structure instead of random texture
* **conv1 filters** — random noise turns into colour / edge detectors
* **vectors** — the 2048 flattened features, the 256-value hidden layer, and the final 2 logits
* **statistics** — mean / std of every layer, untrained vs. trained

In [ ]:
ckpt = torch.load(ckpt_path, map_location="cpu")
for k, v in ckpt.items():
    if k != "model_state_dict":
        describe(k, v)

model, class_names = load_model(ckpt_path, device)
trained_outputs = trace(model, x, device)

show_map_rows(trained_outputs, title="TRAINED feature maps: " + os.path.basename(path))
show_conv1_filters(model, "conv1 filters - trained")
show_vectors(trained_outputs, title="classifier vectors (trained)")

name, conf = predict(model, class_names, path, device, val_t)
print("prediction: {} ({:.2f}) | true: {}".format(name, conf, names[label]))

before, after = layer_stats(untrained_outputs), layer_stats(trained_outputs)
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
for ax, col, title in ((axes[0], 1, "mean of layer output"), (axes[1], 2, "std of layer output")):
    ax.plot([s[col] for s in before], marker="o", markersize=3, label="untrained")
    ax.plot([s[col] for s in after], marker="o", markersize=3, label="trained")
    ax.set_xticks(range(len(before)))
    ax.set_xticklabels([s[0] for s in before], rotation=90, fontsize=7)
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 13. All validation faces + confusion matrix

`predict()` from `bcdl_inference.py` on every validation image: green = correct, red = wrong.
The confusion matrix counts true person (rows) against predicted person (columns).

In [ ]:
idx = {n: l for l, n in class_names.items()}
results = [(p, l) + predict(model, class_names, p, device, val_t) for p, l in val_samples]

show_grid([np.asarray(load_image(p)) for p, *_ in results],
          ["{}\n{} {:.2f}".format(os.path.basename(p), n, c) for p, _, n, c in results],
          colors=["green" if idx[n] == l else "red" for _, l, n, _ in results],
          cols=6, size=2.2, suptitle="validation predictions");

k = len(class_names)
cm = np.zeros((k, k), dtype=int)
for _, l, n, _ in results:
    cm[l, idx[n]] += 1
fig, ax = plt.subplots(figsize=(3.5 + k, 3 + k))
ax.imshow(cm, cmap="Blues")
for i in range(k):
    for j in range(k):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
ticks = [class_names[i] for i in range(k)]
ax.set_xticks(range(k))
ax.set_xticklabels(ticks)
ax.set_yticks(range(k))
ax.set_yticklabels(ticks)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title("accuracy {:.3f}".format(np.trace(cm) / cm.sum()))
plt.tight_layout()
plt.show()

## 14. Interactive explorer

Pick any image (train or validation), any layer with 2-D maps, and how many channels to show.
Each change runs the trained model on that image once. Here every map shares **one** colour scale
(unlike cells 8/12), so you also see which channels are strong and which are almost silent.

In [ ]:
import ipywidgets as widgets

all_samples = val_samples + train_samples
map_layers = [(i, layer, t.shape) for i, layer, t in trained_outputs if t.dim() == 4]


def explore(sample, layer_index, channels):
    p, l = all_samples[sample]
    xx = val_t(load_image(p)).unsqueeze(0).to(device)
    i, layer, t = quiet_trace(model, xx)[layer_index]
    n, conf = predict(model, class_names, p, device, val_t)
    maps = t[0, :channels].numpy()
    lo, hi = maps.min(), maps.max()
    cols = 8
    rows = 1 + (len(maps) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.6), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    axes[0, 0].imshow(np.asarray(load_image(p)))
    axes[0, 0].set_title("{} {}".format("val" if sample < len(val_samples) else "train",
                                        os.path.basename(p)), fontsize=8)
    axes[0, 1].text(0, 0.5, "true: {}\npred: {} ({:.2f})\nlayer #{} {}\noutput {}".format(
        names[l], n, conf, i, short(layer), list(t.shape)), fontsize=9, va="center")
    for c, m in enumerate(maps):
        ax = axes.flat[cols + c]
        ax.imshow(m, cmap="viridis", vmin=lo, vmax=hi, interpolation="nearest")
        ax.set_title("ch {}  max {:.2f}".format(c, m.max()), fontsize=7)
    plt.tight_layout()
    plt.show()


widgets.interact(
    explore,
    sample=widgets.IntSlider(value=0, min=0, max=len(all_samples) - 1, description="image"),
    layer_index=widgets.Dropdown(options=[("#{} {} {}".format(i, short(layer), list(s)), i)
                                          for i, layer, s in map_layers],
                                 value=map_layers[2][0], description="layer"),
    channels=widgets.IntSlider(value=16, min=4, max=128, step=4, description="channels"),
);